In [ ]:
import math
import torch
from torch.utils.data import Dataset,DataLoader
import torch.nn as nn
from torch.nn import functional as F
import pytorch_lightning as L
import pandas as pd
from sklearn.metrics import f1_score



In [ ]:
train_df = pd.read_excel('train.xlsx')
valid_df = pd.read_excel('valid.xlsx')
test_df = pd.read_excel('test.xlsx')
train_df.shape, valid_df.shape, test_df.shape

((5701, 2), (1222, 2), (1222, 2))

In [ ]:
valid_df[:20]

,Tweet,Label,Tweet_clean
0,1 there have been no reported popcorn lung cas...,1,1 there have been no reported popcorn lung cas...
1,she’s a 10 but snorts enough ketamine to kill ...,0,she’s a 10 but snorts enough <DRUGS> to kill a...
2,you can’t overdose on shrooms or acid?,0,you can’t overdose on <DRUGS> or <DRUGS>?
3,"how is this one-day ""transformation"" going to ...",0,"how is this one-day ""transformation"" going to ..."
4,turns out i have adhd and thats why i found m...,1,turns out i have adhd and thats why i found <...
5,i need to stop smoking before work i keep fuck...,2,i need to stop <DRUGS> before work i keep fuck...
6,weirdmicro prompts 421-425 421 - tingling toes...,0,weirdmicro prompts 421-425 421 - tingling toes...
7,appropriate for a guy who has resting stoned f...,0,appropriate for a guy who has resting <DRUGS> ...
8,i got my stuff's from online store they got ls...,1,i got my stuff's from online store they got <D...
9,u deserve lo shunj,0,u deserve lo shunj


In [ ]:
train_df["Label"].value_counts()


,count
Label,
0,2749
1,2005
2,947


In [ ]:
!pip install transformers

In [ ]:
drug_words = [
    "alcohol", "booze", "weed", "cannabis", "marijuana", "pot", "joint", "stoned",
    "methamphetamine", "crystal", "meth",
    "cocaine", "crack",
    "ayahuasca",
    "heroin",
    "e-cigarettes", "vaping", "smoking", "nicotine", "tobacco",
    "lysergic acid diethylamide", "lsd", "acid",
    "psilocybin", "magic mushrooms", "shrooms",
    "salvia",
    "phencyclidine", "pcp", "angel dust",
    "steroids", "anabolic",
    "rohypnol", "flunitrazepam", "roofies",
    "mescaline", "peyote",
    "prescription opioids", "oxy", "percs",
    "dextromethorphan", "dxm",
    "benzos",
    "synthetic cathinones", "bath salts", "flakka",
    "mdma", "ecstasy", "molly",
    "prescription stimulants",
    "gamma-hydroxybutyrate", "ghb",
    "kratom",
    "ketamine",
    "khat"
]

In [ ]:
import re

# Escape words + sort by length (longer phrases first)
escaped_words = sorted(map(re.escape, drug_words), key=len, reverse=True)

pattern = r'\b(' + '|'.join(escaped_words) + r')\b'


In [ ]:
def replace_drug_words(df):
  df["Tweet_clean"] = df["Tweet"].astype(str).str.replace(
    pattern,
    "<DRUGS>",
    flags=re.IGNORECASE,
    regex=True
  )
replace_drug_words(train_df)
replace_drug_words(valid_df)

In [ ]:
# HYPERPARAMETERS ----------------------------------------
max_length = 256 # tokenized text max length

In [ ]:
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
special_tokens_dict = {
    "additional_special_tokens": ["<DRUGS>"]
}
tokenizer.add_special_tokens(special_tokens_dict)

1

In [ ]:
texts = train_df["Tweet_clean"].tolist()

encodings = tokenizer(
    texts,
    truncation=True,
    padding="max_length",
    max_length=max_length,
    return_tensors="pt"
)

encodings["input_ids"].shape

val_texts = valid_df["Tweet_clean"].tolist()
val_labels = valid_df["Label"].values

val_encodings = tokenizer(
    val_texts,
    truncation=True,
    padding="max_length",
    max_length=max_length,
    return_tensors="pt"
)

In [ ]:
for i in range(10):
  tokens = tokenizer.convert_ids_to_tokens(encodings["input_ids"][i])
  print("Original tweet:")
  print(train_df.loc[i, "Tweet_clean"])
  print("Tokens:")
  print(tokens)
  print('\n')

Original tweet:
in order to die from a <DRUGS> overdose you’d have to smoke your body weight worth of <DRUGS> in a single session
Tokens:
['[CLS]', 'in', 'order', 'to', 'die', 'from', 'a', '<DRUGS>', 'overdose', 'you', '’', 'd', 'have', 'to', 'smoke', 'your', 'body', 'weight', 'worth', 'of', '<DRUGS>', 'in', 'a', 'single', 'session', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]'

In [ ]:
encodings["input_ids"][8]

tensor([  101,  1045,  2031,  2699, 30522,  1017,  2335,  1998,  1045,  2031,
         1037,  2261, 21628,  2015,  2035,  2012,  2320,  2012,  1996,  2203,
         1997,  1037,  2095,  2004,  1037,  2210,  7438,  2000,  2870,  2009,
         7126,  2007,  6245,  2009,  1005,  1055,  2025, 26855,  3512,  1045,
         8046, 30522,  1998, 30522,  2000,  2023,  2009,  2003,  7078,  2166,
         5278,   102,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0, 

In [ ]:
print(encodings['attention_mask'].shape)
print(encodings['input_ids'][:5])

torch.Size([5701, 256])
tensor([[  101,  1999,  2344,  ...,     0,     0,     0],
        [  101, 14685, 20920,  ...,     0,     0,     0],
        [  101,  1045,  2342,  ...,     0,     0,     0],
        [  101,  2129,  2055,  ...,     0,     0,     0],
        [  101,  2045,  1005,  ...,     0,     0,     0]])


#### creating the encoder

In [ ]:
from transformers import BertModel

model = BertModel.from_pretrained("bert-base-uncased")

# Resize embeddings to include <DRUGS>
model.resize_token_embeddings(len(tokenizer))

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(30523, 768, padding_idx=0)

In [ ]:
!pip install pytorch_lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.5/849.5 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 56.4 MB/s eta 0:00:00


In [ ]:
class TokenizedDataset(Dataset):
    def __init__(self, encodings, labels):
        self.input_ids = encodings["input_ids"]
        self.attn_mask = encodings["attention_mask"]
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            self.input_ids[idx].long(),   # (256)
            self.attn_mask[idx].long(),   # (256)
            self.labels[idx]              # ()
        )

In [ ]:
labels = train_df["Label"].values
val_labels = valid_df["Label"].values

train_dataset = TokenizedDataset(encodings, labels)
val_dataset   = TokenizedDataset(val_encodings, val_labels)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)


In [ ]:
class TransformerEmbedding(nn.Module):
    def __init__(self, vocab_size, hidden_size, max_len):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, hidden_size)
        self.pos_emb = nn.Embedding(max_len, hidden_size)
        self.norm = nn.LayerNorm(hidden_size)

    def forward(self, input_ids):
        B, L = input_ids.shape
        positions = torch.arange(L, device=input_ids.device).unsqueeze(0)
        x = self.token_emb(input_ids) + self.pos_emb(positions)
        return self.norm(x)  # (B, L, D)

In [ ]:
def scaled_dot_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)

    attn = torch.softmax(scores, dim=-1)
    return attn @ V

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, hidden_size, num_heads):
        super().__init__()
        assert hidden_size % num_heads == 0

        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads

        self.qkv = nn.Linear(hidden_size, hidden_size * 3)
        self.out = nn.Linear(hidden_size, hidden_size)

    def forward(self, x, mask):
        B, L, D = x.shape

        qkv = self.qkv(x)
        Q, K, V = qkv.chunk(3, dim=-1)

        Q = Q.view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, L, self.num_heads, self.head_dim).transpose(1, 2)

        out = scaled_dot_attention(Q, K, V, mask)
        out = out.transpose(1, 2).contiguous().view(B, L, D)
        return self.out(out)

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, hidden_size, ffn_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_size, ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim, hidden_size)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, hidden_size, num_heads, ffn_dim):
        super().__init__()
        self.attn = MultiHeadSelfAttention(hidden_size, num_heads)
        self.ffn = FeedForward(hidden_size, ffn_dim)
        self.norm1 = nn.LayerNorm(hidden_size)
        self.norm2 = nn.LayerNorm(hidden_size)

    def forward(self, x, mask):
        x = self.norm1(x + self.attn(x, mask))
        x = self.norm2(x + self.ffn(x))
        return x

In [ ]:
class TransformerEncoder(nn.Module):
    def __init__(self, num_layers, hidden_size, num_heads, ffn_dim):
        super().__init__()
        self.layers = nn.ModuleList([
            EncoderLayer(hidden_size, num_heads, ffn_dim)
            for _ in range(num_layers)
        ])

    def forward(self, x, mask):
        for layer in self.layers:
            x = layer(x, mask)
        return x

In [ ]:
class EncoderClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        max_len=256,
        hidden_size=128,
        num_heads=4,
        num_layers=2,
        ffn_dim=512,
        num_classes=3
    ):
        super().__init__()

        self.embedding = TransformerEmbedding(vocab_size, hidden_size, max_len)
        self.encoder = TransformerEncoder(num_layers, hidden_size, num_heads, ffn_dim)
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        """
        input_ids      : (B, L)
        attention_mask : (B, L)
        """
        mask = attention_mask.unsqueeze(1).unsqueeze(2)  # (B,1,1,L)

        x = self.embedding(input_ids)     # (B,L,D)
        x = self.encoder(x, mask)         # (B,L,D)
        cls = x[:, 0, :]                  # (B,D)

        return self.classifier(cls)       # (B,3)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = EncoderClassifier(
    vocab_size=len(tokenizer),
    hidden_size=128,
    num_heads=4,
    num_layers=2,
    ffn_dim=512,
    num_classes=3
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()


In [ ]:
epochs = 20

for epoch in range(epochs):
    # -------- TRAIN --------
    model.train()
    train_loss = 0.0

    for batch in train_loader:
        input_ids, attention_mask, labels = batch

        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(input_ids, attention_mask)  # (B,3)
        loss = criterion(logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # -------- VALIDATION --------
    model.eval()
    val_loss = 0.0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in val_loader:
            input_ids, attention_mask, labels = batch

            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)

            val_loss += loss.item()

            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)

    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    weighted_f1 = f1_score(all_labels, all_preds, average="weighted")



    print(
      f"Epoch [{epoch+1}/{epochs}] | "
      f"Train Loss: {avg_train_loss:.4f} | "
      f"Val Loss: {avg_val_loss:.4f} | "
      f"Macro F1: {macro_f1:.4f} | "
      f"Weighted F1: {weighted_f1:.4f}"
    )


Epoch [1/20] | Train Loss: 0.9144 | Val Loss: 0.9600 | Macro F1: 0.3929 | Weighted F1: 0.4746
Epoch [2/20] | Train Loss: 0.8745 | Val Loss: 0.9353 | Macro F1: 0.4447 | Weighted F1: 0.5272
Epoch [3/20] | Train Loss: 0.8406 | Val Loss: 0.9360 | Macro F1: 0.4917 | Weighted F1: 0.5566
Epoch [4/20] | Train Loss: 0.8135 | Val Loss: 0.9391 | Macro F1: 0.4734 | Weighted F1: 0.5407
Epoch [5/20] | Train Loss: 0.7826 | Val Loss: 0.9291 | Macro F1: 0.5011 | Weighted F1: 0.5678
Epoch [6/20] | Train Loss: 0.7497 | Val Loss: 0.9420 | Macro F1: 0.4791 | Weighted F1: 0.5538
Epoch [7/20] | Train Loss: 0.7143 | Val Loss: 0.9653 | Macro F1: 0.5044 | Weighted F1: 0.5571
Epoch [8/20] | Train Loss: 0.6824 | Val Loss: 1.0129 | Macro F1: 0.5151 | Weighted F1: 0.5556
Epoch [9/20] | Train Loss: 0.6440 | Val Loss: 0.9911 | Macro F1: 0.4923 | Weighted F1: 0.5498
Epoch [10/20] | Train Loss: 0.6068 | Val Loss: 1.0280 | Macro F1: 0.4969 | Weighted F1: 0.5482
Epoch [11/20] | Train Loss: 0.5780 | Val Loss: 1.0846 | Mac